# Introduction to Qdrant Cloud

## Load Keys in the Env.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## Connect

In [2]:
import os
from qdrant_client import QdrantClient

qdrant = QdrantClient(
    url = os.environ["QDRANT_URL"],
    api_key= os.environ["QDRANT_API_KEY"]
)

print(f"Existing collections: {[c.name for c in qdrant.get_collections().collections]}")

Existing collections: []


## Dummy Dataset

In [4]:
movies = [
    {
        "title": "The Matrix",
        "text": "A hacker discovers reality is a simulation and joins a rebellion against the machines that run it.",
    },
    {
        "title": "Finding Nemo",
        "text": "An anxious clownfish crosses the ocean to rescue his son from a dentist's aquarium.",
    },
    {
        "title": "Interstellar",
        "text": "A former astronaut pilots a wormhole mission to find humanity a new home among the stars.",
    },
    {
        "title": "Ratatouille",
        "text": "A Parisian rat with a refined palate secretly cooks gourmet meals through a young kitchen hand.",
    },
    {
        "title": "Mad Max: Fury Road",
        "text": "A drifter and a rebel war captain flee a tyrant across a post-apocalyptic desert in a roaring convoy.",
    },
    {
        "title": "Spirited Away",
        "text": "A young girl wanders into a spirit world bathhouse and must work to free her transformed parents.",
    },
]

print(f"{len(movies)} movies ready.")

6 movies ready.


## Embed With Cohere

In [5]:
import cohere

co = cohere.ClientV2(
    api_key=os.environ["COHERE_API_KEY"],
)

embed_response = co.embed(
    model="embed-v4.0",
    input_type="search_document",
    embedding_types=["float"],
    texts=[m["text"] for m in movies],
)

vectors = embed_response.embeddings.float

VECTOR_SIZE = len(vectors[0])
print(f"Embedded {len(vectors)} movies. Vector dimensions: {VECTOR_SIZE}")

Embedded 6 movies. Vector dimensions: 1536


## Create Collection and Upload "Points"

In [8]:
from qdrant_client.models import Distance, VectorParams, PointStruct

COLLECTION_NAME = "week0_movies"

# in prod just use `create_collection` once.
qdrant.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)

qdrant.upsert(
    collection_name=COLLECTION_NAME,
    points = [
        PointStruct(id=i, vector=vec, payload=movie) for i, (movie, vec) in enumerate(zip(movies, vectors))
    ]
)

info = qdrant.get_collection(COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' now has {info.points_count} points.")

/var/folders/sq/n6k1jqxx3hg9141zty3_ggyw0000gr/T/ipykernel_71774/3296110403.py:6: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


Collection 'week0_movies' now has 6 points.


## Search

In [11]:
query = "a heartwarming animated story about food"

query_vec = co.embed(
    model="embed-v4.0",
    input_type="search_query",
    embedding_types=["float"],
    texts=[query]
).embeddings.float[0]

hits = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vec,
    limit=3
).points

print(f"Query: {query!r}\n")
for h in hits:
    print(f"[{h.score:.3f}] {h.payload['title']} — {h.payload['text']}")

Query: 'a heartwarming animated story about food'

[0.424] Ratatouille — A Parisian rat with a refined palate secretly cooks gourmet meals through a young kitchen hand.
[0.279] Spirited Away — A young girl wanders into a spirit world bathhouse and must work to free her transformed parents.
[0.276] Finding Nemo — An anxious clownfish crosses the ocean to rescue his son from a dentist's aquarium.
